# 07 - RAG with LangChain

You built RAG by hand: chunking, embeddings, vector search, prompt assembly,
generation. Frameworks like LangChain package those same steps behind
ready-made components. This notebook rebuilds notebook 06 with LangChain and
maps every component back to the code you already wrote.

**What you will learn**

- The LangChain equivalents of each hand-built piece
- How to compose them into a chain
- When a framework helps, and when it gets in the way

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")
print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


## The component map

| Hand-built (notebooks 02 to 06) | LangChain |
|---|---|
| chunk_text() function | RecursiveCharacterTextSplitter |
| client.embeddings.create(...) | OpenAIEmbeddings |
| ChromaDB collection + query | Chroma vector store + retriever |
| build_prompt() f-string | ChatPromptTemplate |
| client.chat.completions.create(...) | ChatOpenAI |
| calling the steps in order | the pipe operator |

## Load and split the documents

LangChain passes text around as Document objects: text plus a metadata
dictionary. Its go-to splitter is RecursiveCharacterTextSplitter, a smarter
cousin of our chunk_text: it tries to split on paragraph breaks first, then
sentences, then words, so chunks break at natural boundaries when possible.

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = [
    Document(page_content=path.read_text(), metadata={"source": path.name})
    for path in sorted(Path("../data").glob("*.txt"))
]

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents became {len(chunks)} chunks.")

6 documents became 13 chunks.


## Embed and store

Chroma.from_documents does in one line what took us a setup cell plus an
upsert in notebook 05: create a collection, embed every chunk, store text
and metadata. Metadata (like source) is carried along automatically.

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="aurora_langchain",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

A retriever is a tiny wrapper around the vector store with one job:
take a question, return relevant Documents. It is our retrieve() function
as an object:

In [4]:
for doc in retriever.invoke("How many vacation days do employees get?"):
    print(f"[{doc.metadata['source']}] {doc.page_content[:80]}...")

[03-leave-policy.txt] Public holidays:
Each office follows its local public holiday calendar. In addit...
[03-leave-policy.txt] Aurora Dynamics - Employee Handbook: Leave Policy

This policy applies to all fu...
[04-remote-work-policy.txt] Aurora Dynamics - Employee Handbook: Remote Work Policy

Aurora Dynamics uses a ...
[04-remote-work-policy.txt] Remote work outside the employee's country of employment is limited to
30 days p...


## Prompt and model

The same system prompt as notebook 06, as a ChatPromptTemplate with two
placeholders, plus the LLM wrapper:

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about Aurora Dynamics using ONLY the provided context. "
     "If the context does not contain the answer, reply exactly: "
     "\"I don't know based on the available documents.\" "
     "Never use outside knowledge. Mention the source file(s) you used."),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Composing the chain

LangChain's pipe operator (|) connects components: the output of each step
flows into the next, exactly like our ask() function called retrieve, then
build_prompt, then the LLM, in order.

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[source: {d.metadata['source']}]\n{d.page_content}" for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Reading it top to bottom: the question goes to the retriever (whose
documents are formatted into a context string) and is also passed through
unchanged; both fill the prompt template; the prompt goes to the LLM; the
output parser extracts plain text from the reply.

## Trying it out

Same questions as notebook 06, same behavior:

In [7]:
print(rag_chain.invoke("How many days of paid annual leave do employees get?"))

Employees receive 24 days of paid annual leave per calendar year. [source: 03-leave-policy.txt]


In [8]:
print(rag_chain.invoke("A robot is blinking orange in a corridor. What should I do?"))

To resolve the issue of the robot blinking orange in a corridor, you should wipe the lidar dome with the microfiber cloth from the maintenance kit. If the robot still blinks orange after cleaning, restart it by holding the power button for 8 seconds. If the issue repeats more than twice in one day, escalate to level 2 support. 

(Source: 05-support-runbook.txt)


In [9]:
print(rag_chain.invoke("What color is the Carrier X2?"))

I don't know based on the available documents.


The honesty test still passes, because the prompt rules came along.

## Framework or no framework?

You have now written the same system twice. A fair comparison:

**LangChain helps because**

- less code for standard patterns, batteries included
- swapping components is one line: a different vector store, LLM provider,
  or splitter without rewriting the pipeline
- a huge ecosystem of loaders (PDF, HTML, Notion, and hundreds more)

**Plain Python helps because**

- every line is yours: debugging is straightforward
- no framework abstractions to learn on top of the RAG concepts
- fewer dependencies to break on upgrade

A good rule: prototype with a framework if its components fit your need
exactly, but make sure you understand the loop underneath, which is
precisely what notebooks 01 to 06 gave you. Many production teams write the
core RAG loop in plain Python and pull in libraries only for specific parts
like document parsing.

## Exercise

1. Change k from 4 to 2 in the retriever and re-run the questions. Any
   difference?
2. Change chunk_size in the splitter to 300 and rebuild the vector store.
   How does the chunk count change?
3. Ask a question that needs information from two different files and check
   whether both sources are mentioned.

In [10]:
# Try the exercise here


## Where to go next

You know the core RAG loop. Natural next steps, roughly in order of impact:

- **Better retrieval**: hybrid search (combine keyword and vector search),
  reranking retrieved chunks with a dedicated model
- **Better questions**: query rewriting, letting the LLM reformulate vague
  questions before retrieval
- **Evaluation**: measuring answer quality systematically, for example with
  RAGAS
- **Real documents**: PDF parsing, tables, images
- **Agentic RAG**: letting the model decide when and what to retrieve,
  possibly over multiple steps

Congratulations on finishing the course.